# Model Evaluation & SHAP Analysis
**Goal**: Compare all 5 models, visualize metrics, and use SHAP for XGBoost, Random Forest, and Logistic Regression.

## Sections
1. Load models & data
2. Compute metrics (Accuracy, Precision, Recall, F1, F2, ROC-AUC)
3. Comparison bar chart
4. Confusion matrices
5. ROC curves
6. Classification reports
7. SHAP analysis (XGBoost, Random Forest, Logistic Regression)
8. Assessment & notes for improvement

In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, fbeta_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
print('Imports OK')

## 1. Load Models & Data

In [ ]:
import sys; sys.path.insert(0, os.path.dirname(os.getcwd())) # needed for jupyter relative imports
from utils.data_loader import (
    load_processed_data, split_and_scale,
    load_model, load_scaler, load_feature_names
)

X, y = load_processed_data()
X_train, X_test, y_train, y_test, scaler, feature_names = split_and_scale(X, y, save=False)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Features ({len(feature_names)}): {feature_names}')
print(f'Test positive rate: {y_test.mean():.2%}')

In [ ]:
MODEL_NAMES = {
    'XGBoost':              'xgboost',
    'Random Forest':        'random_forest',
    'Logistic Regression':  'logistic_regression',
    'SVM':                  'svm',
    'KNN':                  'knn',
}

models = {}
for display_name, file_name in MODEL_NAMES.items():
    models[display_name] = load_model(file_name)
    print(f'Loaded: {display_name}')

print(f'\nAll {len(models)} models loaded.')

## 2. Compute Metrics

In [ ]:
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = model.decision_function(X_test)
    
    results.append({
        'Model':     name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1 Score':  f1_score(y_test, y_pred),
        'F2 Score':  fbeta_score(y_test, y_pred, beta=2),
        'ROC-AUC':   roc_auc_score(y_test, y_prob),
    })

df_results = pd.DataFrame(results).set_index('Model')
print('Model Comparison:')
df_results.style.highlight_max(axis=0, color='lightgreen').format('{:.4f}')

## 3. Comparison Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
df_results.plot(kind='bar', ax=ax, rot=0, width=0.75)
ax.set_ylim(0.55, 0.85)
ax.set_ylabel('Score')
ax.set_title('Model Comparison - All Metrics', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', framealpha=0.9)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=7, padding=2)
plt.tight_layout()
plt.show()

## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Risk', 'At Risk'],
                yticklabels=['No Risk', 'At Risk'])
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual' if ax == axes[0] else '')
plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
for name, model in models.items():
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = model.decision_function(X_test)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random (AUC = 0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - All Models', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Classification Reports

In [ ]:
for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f'\n{"=" * 55}')
    print(f'  {name}')
    print(f'{"=" * 55}')
    print(classification_report(y_test, y_pred, target_names=['No Risk', 'At Risk']))

---
## 7. SHAP Analysis (XGBoost, Random Forest, Logistic Regression)

Explainer types:
- **TreeExplainer**: XGBoost, Random Forest (exact, fast)
- **LinearExplainer**: Logistic Regression (exact for linear models)

> SVM and KNN are excluded from SHAP analysis because KernelExplainer is very slow and yields less reliable results.

In [ ]:
import shap
print(f'SHAP version: {shap.__version__}')

### 7.1 XGBoost SHAP

In [ ]:
xgb_explainer = shap.TreeExplainer(models['XGBoost'])
xgb_shap = xgb_explainer.shap_values(X_test)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
plt.sca(axes[0])
shap.summary_plot(xgb_shap, X_test, feature_names=feature_names, show=False)
axes[0].set_title('XGBoost - SHAP Beeswarm', fontweight='bold')
plt.sca(axes[1])
shap.summary_plot(xgb_shap, X_test, feature_names=feature_names, plot_type='bar', show=False)
axes[1].set_title('XGBoost - Mean |SHAP|', fontweight='bold')
plt.tight_layout()
plt.show()

### 7.2 Random Forest SHAP

In [ ]:
rf_explainer = shap.TreeExplainer(models['Random Forest'])
X_test_sample = X_test.sample(500, random_state=42)
rf_shap = rf_explainer.shap_values(X_test_sample)
if isinstance(rf_shap, list):
    rf_shap = rf_shap[1]

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
plt.show() # clear previous if any
shap.summary_plot(rf_shap, X_test_sample, feature_names=feature_names)
shap.summary_plot(rf_shap, X_test_sample, feature_names=feature_names, plot_type='bar')

### 7.3 Logistic Regression SHAP

In [ ]:
lr_explainer = shap.LinearExplainer(models['Logistic Regression'], X_train)
lr_shap = lr_explainer.shap_values(X_test)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
plt.sca(axes[0])
shap.summary_plot(lr_shap, X_test, feature_names=feature_names, show=False)
axes[0].set_title('Logistic Regression - SHAP Beeswarm', fontweight='bold')
plt.sca(axes[1])
shap.summary_plot(lr_shap, X_test, feature_names=feature_names, plot_type='bar', show=False)
axes[1].set_title('Logistic Regression - Mean |SHAP|', fontweight='bold')
plt.tight_layout()
plt.show()

### 7.4 Cross-Model Feature Importance Comparison

In [ ]:
importance_data = {}
all_shap = {
    'XGBoost':    (xgb_shap, X_test),
    'Random Forest': (rf_shap, X_test_sample),
    'Logistic Regression': (lr_shap, X_test),
}

for name, (sv, _) in all_shap.items():
    if isinstance(sv, list):
        sv = sv[1]
    if sv.ndim == 3:
        sv = sv[:, :, 1]
    importance_data[name] = np.abs(sv).mean(axis=0)

df_importance = pd.DataFrame(importance_data, index=feature_names)
df_importance_norm = df_importance / df_importance.max()
df_importance_norm['Average'] = df_importance_norm.mean(axis=1)
df_importance_norm = df_importance_norm.sort_values('Average', ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
df_importance_norm.drop(columns='Average').plot(kind='barh', ax=ax, width=0.8, alpha=0.85)
ax.set_xlabel('Normalized Mean |SHAP Value|', fontsize=12)
ax.set_title('Feature Importance - XGBoost vs RF vs LR', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

print('\nTop 10 features by average importance:')
df_importance_norm[['Average']].sort_values('Average', ascending=False).head(10)

---
## 8. Assessment & Notes

### Observations

*(Fill in after running)*

1. **Best overall model**: Compare accuracy, F1, F2, ROC-AUC
2. **Recall vs Precision**: Which model has highest recall?
3. **Feature consistency**: Do all 3 models agree on top features?
4. **Model for xAI**: **Logistic Regression** is recommended for counterfactual explanations:
   - Linear = smooth, predictable CF changes
   - Best/tied-best F2 score (recall)
   - Fast CF optimization

### Potential Improvements

- Hyperparameter tuning (GridSearchCV / Optuna)
- More granular `class_weight` tuning
- Threshold tuning for best F2 or recall
- Cross-validation (StratifiedKFold)
- Drop features with near-zero SHAP importance